# UCS420 Assignment 10: Healthcare Cognitive Assistant
**Google Colab notebook | NLP + Taxonomy + Ontology + Machine Learning + Rule-based reasoning**




In [ ]:
%pip -q install scikit-learn pandas matplotlib seaborn nltk ipywidgets

In [ ]:
import re, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from IPython.display import display, Markdown
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
for item, path in [('stopwords','corpora/stopwords'),('wordnet','corpora/wordnet'),('omw-1.4','corpora/omw-1.4')]:
    try: nltk.data.find(path)
    except LookupError: nltk.download(item, quiet=True)
DOMAIN = "diabetes"  # CHANGE TO "heart" for EVEN roll numbers
assert DOMAIN in ("diabetes", "heart")
SEED=42
random.seed(SEED)

## 1. Labelled corpus — 64 queries across 8 intents per domain

In [ ]:
# Eight intent categories; each has eight independently written training examples.
CORPORA = {
"diabetes": {
"overview": ["What is diabetes?", "Explain diabetes in simple terms", "What does having diabetes mean?", "Tell me about diabetes", "What is high blood sugar disease?", "Give an overview of diabetes", "What are the types of diabetes?", "What happens in diabetes?"],
"symptoms": ["What are common diabetes symptoms?", "Why am I thirsty all the time with diabetes?", "What are signs of high blood glucose?", "Can frequent urination indicate diabetes?", "Tell me warning signs of diabetes", "What symptoms can diabetes cause?", "What are early signs of diabetes?", "Does blurry vision occur with diabetes?"],
"risk_factors": ["Who is at risk of type 2 diabetes?", "What increases diabetes risk?", "Does family history matter for diabetes?", "Can inactivity raise diabetes risk?", "What are diabetes risk factors?", "How does age affect diabetes risk?", "Does excess weight increase diabetes risk?", "Who is more likely to develop diabetes?"],
"screening": ["How is diabetes diagnosed?", "What is an A1C test?", "What blood tests check diabetes?", "How does fasting glucose testing work?", "When should someone get screened for diabetes?", "What is the purpose of glucose testing?", "How do doctors test for diabetes?", "What does an oral glucose tolerance test measure?"],
"nutrition": ["What foods support diabetes management?", "How do carbohydrates affect blood sugar?", "Can you explain balanced meals for diabetes?", "What is the diabetes plate method?", "Should I limit sugary drinks with diabetes?", "How can fiber help with diabetes meals?", "What are healthy snacks for diabetes?", "How do I plan meals with diabetes?"],
"activity": ["Does exercise help manage diabetes?", "How can walking affect blood sugar?", "What physical activity is good for diabetes?", "Why is regular movement important for diabetes?", "Can strength training help with diabetes?", "How do I start exercising with diabetes?", "What are benefits of exercise for glucose?", "How often should people with diabetes be active?"],
"medication": ["What medicines are used for diabetes?", "What is insulin used for?", "Explain metformin for diabetes", "Why do some people need insulin?", "What are common diabetes medication types?", "Can I stop diabetes medication myself?", "How do diabetes medicines help glucose?", "What should I ask about diabetes prescriptions?"],
"prevention": ["How can type 2 diabetes be prevented?", "Can lifestyle changes lower diabetes risk?", "How does healthy eating prevent diabetes?", "Does regular exercise help prevent diabetes?", "What is prediabetes?", "Can prediabetes be reversed?", "How can I reduce my chance of diabetes?", "What habits support diabetes prevention?"]
},
"heart": {
"overview": ["What is cardiovascular health?", "Explain heart disease", "What does heart health mean?", "Tell me about cardiovascular disease", "What is coronary artery disease?", "Give an overview of heart health", "How does the heart work?", "What are common types of heart disease?"],
"symptoms": ["What are warning signs of heart disease?", "What symptoms can a heart attack cause?", "Can shortness of breath indicate a heart problem?", "What does chest discomfort mean?", "What are common cardiovascular symptoms?", "What signs of heart trouble should I know?", "Can unusual fatigue be related to heart disease?", "What are signs of a possible cardiac problem?"],
"risk_factors": ["What increases heart disease risk?", "How does smoking affect heart health?", "Does family history influence heart risk?", "Is high cholesterol a heart risk factor?", "What are cardiovascular risk factors?", "How does diabetes affect heart disease risk?", "Can stress influence heart health?", "Who is at higher risk for heart disease?"],
"screening": ["What tests check heart health?", "How is blood pressure measured?", "What is a cholesterol panel?", "Why do doctors order an ECG?", "How do clinicians assess heart risk?", "When should blood pressure be checked?", "What is a lipid profile?", "What does cardiovascular screening involve?"],
"nutrition": ["What foods are good for heart health?", "How does sodium affect blood pressure?", "What is a heart healthy diet?", "Should I reduce saturated fat?", "How do fruits and vegetables help the heart?", "Explain the DASH eating plan", "What are healthy fats for the heart?", "How can I eat to support cardiovascular health?"],
"activity": ["How does exercise help the heart?", "Is walking good for cardiovascular fitness?", "What physical activity supports heart health?", "Why is aerobic exercise important?", "How do I begin exercising for heart health?", "Can regular movement lower heart risk?", "How does fitness affect blood pressure?", "What are the heart benefits of exercise?"],
"medication": ["What medicines treat heart conditions?", "What are statins used for?", "Why are blood pressure medicines prescribed?", "What do beta blockers do?", "What are common cardiovascular medications?", "Can I stop my heart medicine myself?", "What should I know about prescribed statins?", "How do heart medications reduce risk?"],
"prevention": ["How can I prevent heart disease?", "What habits protect the heart?", "Can quitting smoking reduce heart risk?", "How does sleep affect cardiovascular health?", "How can I lower my heart disease risk?", "Why is blood pressure control important?", "What lifestyle changes support heart health?", "How can I keep my heart healthy?"]
}}
INTENTS=list(CORPORA[DOMAIN])
df=pd.DataFrame([(q,intent) for intent,qs in CORPORA[DOMAIN].items() for q in qs],columns=["query","intent"])
print("Domain:",DOMAIN,"| queries:",len(df),"| intents:",df.intent.nunique())
display(df.groupby("intent").size().rename("query_count").to_frame())
display(df.head(10))

## 2. Hierarchical taxonomy — 4 major categories, 2 subcategories each

In [ ]:
TAXONOMY={
    "Understanding": {"Basics":"overview", "Signs and symptoms":"symptoms"},
    "Risk and assessment": {"Risk factors":"risk_factors", "Tests and screening":"screening"},
    "Daily management": {"Nutrition":"nutrition", "Physical activity":"activity"},
    "Treatment and prevention": {"Medicines":"medication", "Prevention":"prevention"}
}
for major,subs in TAXONOMY.items():
    print(major)
    for sub,intent in subs.items(): print("  └─",sub,"→",intent)
assert len(TAXONOMY)>=4 and all(len(v)>=2 for v in TAXONOMY.values())

## 3. Domain ontology — concepts and relationships

In [ ]:
ONTOLOGIES={
"diabetes": [
("Diabetes","involves","Blood glucose"),("Insulin","regulates","Blood glucose"),
("Pancreas","produces","Insulin"),("Type 2 diabetes","is_a","Diabetes"),
("Type 1 diabetes","is_a","Diabetes"),("A1C test","measures","Average blood glucose"),
("Fasting glucose test","measures","Blood glucose"),("Physical activity","supports","Glucose management"),
("Balanced nutrition","supports","Glucose management"),("Medication","supports","Glucose management"),
("Family history","increases_risk_of","Type 2 diabetes"),("Prediabetes","increases_risk_of","Type 2 diabetes")
],
"heart": [
("Heart","pumps","Blood"),("Arteries","carry","Blood"),("High blood pressure","increases_risk_of","Heart disease"),
("High LDL cholesterol","increases_risk_of","Heart disease"),("Smoking","increases_risk_of","Heart disease"),
("Diabetes","increases_risk_of","Heart disease"),("Lipid panel","measures","Cholesterol"),
("Blood pressure test","measures","Blood pressure"),("Exercise","supports","Heart health"),
("Heart-healthy nutrition","supports","Heart health"),("Statins","lower","LDL cholesterol"),
("Coronary artery disease","is_a","Heart disease")
]}
triples=ONTOLOGIES[DOMAIN]
concepts=set([v for a,r,b in triples for v in (a,b)])
print("Concepts:",len(concepts),"| Relationships:",len(triples))
display(pd.DataFrame(triples,columns=["subject","relation","object"]))
assert len(concepts)>=8 and len(triples)>=10

## 4. NLP preprocessing — lowercase, tokenization, stopword removal, lemmatization

In [ ]:
stop_words=set(stopwords.words("english"))
lemmatizer=WordNetLemmatizer()
def preprocess(text):
    lowered=text.lower()
    tokens=re.findall(r"[a-z]+",lowered)
    filtered=[w for w in tokens if w not in stop_words]
    lemmas=[lemmatizer.lemmatize(w, pos="v") for w in filtered]
    lemmas=[lemmatizer.lemmatize(w, pos="n") for w in lemmas]
    return " ".join(lemmas)
example="What medicines are used for managing diabetes?"
print("Original:",example,"\nProcessed:",preprocess(example))
df["processed"]=df["query"].map(preprocess)

## 5. TF-IDF feature extraction and Logistic Regression intent classification

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(df["processed"],df["intent"],test_size=.25,random_state=SEED,stratify=df["intent"])
model=Pipeline([
    ("tfidf",TfidfVectorizer(ngram_range=(1,2),sublinear_tf=True)),
    ("classifier",LogisticRegression(max_iter=2000,random_state=SEED))
])
model.fit(X_train,y_train)
pred=model.predict(X_test)
print("TF-IDF features:",len(model.named_steps["tfidf"].get_feature_names_out()))
print("Training examples:",len(X_train),"| Test examples:",len(X_test))

## 6. Evaluation — accuracy, precision, recall, F1 and confusion matrix

In [ ]:
metrics={
"Accuracy":accuracy_score(y_test,pred),
"Precision (macro)":precision_score(y_test,pred,average="macro",zero_division=0),
"Recall (macro)":recall_score(y_test,pred,average="macro",zero_division=0),
"F1 (macro)":f1_score(y_test,pred,average="macro",zero_division=0)
}
display(pd.DataFrame(metrics.items(),columns=["Metric","Score"]))
print(classification_report(y_test,pred,zero_division=0))
cm=confusion_matrix(y_test,pred,labels=INTENTS)
plt.figure(figsize=(9,6))
sns.heatmap(cm,annot=True,fmt="d",xticklabels=INTENTS,yticklabels=INTENTS,cmap="Blues")
plt.title("Intent Classification — Confusion Matrix")
plt.xlabel("Predicted intent");plt.ylabel("Actual intent")
plt.tight_layout();plt.show()

## 7. Domain knowledge base — information for every intent

In [ ]:
KNOWLEDGE={
"diabetes":{
"overview":"Diabetes is a condition in which blood glucose is too high. Type 1 involves little or no insulin production; type 2 commonly involves insulin resistance. A clinician can explain individual diagnosis and management.",
"symptoms":"Possible symptoms include increased thirst, frequent urination, unexplained weight loss, fatigue and blurry vision. Symptoms alone cannot confirm diabetes; seek medical evaluation.",
"risk_factors":"Type 2 diabetes risk factors include family history, age, physical inactivity, and certain metabolic conditions. Risk varies by person; discuss screening with a clinician.",
"screening":"Clinicians may use A1C, fasting plasma glucose, or an oral glucose tolerance test. Interpretation and confirmation depend on clinical circumstances.",
"nutrition":"A balanced eating pattern with vegetables, fiber-rich foods, suitable carbohydrate portions and fewer sugary drinks may help manage glucose. Individual needs differ.",
"activity":"Regular physical activity can support glucose management. Start at a suitable level and ask a clinician about precautions, particularly if using glucose-lowering medicine.",
"medication":"Treatment may involve insulin or other prescribed medications such as metformin. Never start, stop or adjust doses based on this chatbot; consult your prescriber.",
"prevention":"Healthy eating, regular movement and weight management when appropriate can lower type 2 diabetes risk. Type 1 diabetes is not generally preventable through lifestyle changes."
},
"heart":{
"overview":"Heart health concerns the heart and blood vessels. Cardiovascular diseases include coronary artery disease and other conditions affecting circulation.",
"symptoms":"Possible warning signs include chest discomfort, shortness of breath and unusual fatigue. Sudden or severe symptoms may require emergency care.",
"risk_factors":"Risk factors include smoking, high blood pressure, high LDL cholesterol, diabetes, family history and physical inactivity.",
"screening":"Assessment may include blood pressure measurement, a cholesterol panel and other tests selected by a clinician. ECGs are used for specific clinical questions.",
"nutrition":"A heart-healthy pattern emphasizes vegetables, fruits, whole grains, legumes and unsaturated fats while limiting excess sodium and saturated fat.",
"activity":"Regular aerobic activity and appropriate strengthening can support cardiovascular health. Ask a clinician about exercise restrictions if you have symptoms or known disease.",
"medication":"Depending on the condition, clinicians may prescribe blood-pressure medicines, statins or other treatments. Do not change prescriptions without medical guidance.",
"prevention":"Avoid tobacco, stay active, follow a balanced diet, sleep adequately and manage blood pressure, cholesterol and diabetes with appropriate medical care."
}}
assert set(KNOWLEDGE[DOMAIN])==set(INTENTS)
display(pd.DataFrame(KNOWLEDGE[DOMAIN].items(),columns=["Intent","Knowledge response"]))

## 8. Reasoning rules and safety safeguards

The assistant checks emergency signals **before** the classifier. It does not attempt to diagnose from symptoms. A classifier confidence threshold is a routing heuristic, **not a medically calibrated probability**.

In [ ]:
EMERGENCY_PATTERNS=[
r"\b(chest pain|crushing chest|severe chest pressure|heart attack|stroke|face droop|slurred speech|one sided weakness)\b",
r"\b(can'?t breathe|cannot breathe|severe difficulty breathing|unconscious|passed out|seizure|severe confusion)\b",
r"\b(diabetic ketoacidosis|dka|vomiting with high blood sugar)\b"
]
UNSUPPORTED_PATTERNS=[
r"\b(dental|toothache|braces|acne|eczema|fracture|broken bone|pregnancy|migraine|eye surgery)\b"
]
PERSONAL_MEDICAL_PATTERNS=[
r"\b(how much (insulin|metformin|medicine)|what dose|change my dose|stop taking|diagnose me|interpret my lab)\b"
]
DOMAIN_TERMS={
"diabetes":{"diabetes","diabetic","glucose","sugar","insulin","a1c","metformin","prediabetes","carbohydrate","blood"},
"heart":{"heart","cardiac","cardiovascular","cholesterol","pressure","statin","artery","blood","ecg","coronary"}
}
CONFIDENCE_THRESHOLD=0.30
SIMILARITY_THRESHOLD=0.13

def assistant(query, show_details=True):
    q=query.strip()
    if not q:
        return {"rule":"empty_input","intent":None,"confidence":0.0,"response":"Please enter a healthcare question."}
    # Rule 1: emergency escalation
    if any(re.search(p,q.lower()) for p in EMERGENCY_PATTERNS):
        return {"rule":"emergency_escalation","intent":None,"confidence":1.0,
                "response":"This may be a medical emergency. Call your local emergency number now or seek emergency care immediately. Do not wait for a chatbot response."}
    # Rule 2: personal medical decisions
    if any(re.search(p,q.lower()) for p in PERSONAL_MEDICAL_PATTERNS):
        return {"rule":"personal_medical_decision","intent":None,"confidence":1.0,
                "response":"I cannot diagnose you, interpret personal test results, or recommend medication doses. Please contact a qualified healthcare professional."}
    # Rule 3: clearly unsupported topics
    if any(re.search(p,q.lower()) for p in UNSUPPORTED_PATTERNS):
        return {"rule":"unsupported_query","intent":None,"confidence":0.0,
                "response":f"I only cover general {DOMAIN} health information. Please consult an appropriate healthcare professional for this topic."}
    processed=preprocess(q)
    vec=model.named_steps["tfidf"].transform([processed])
    if vec.nnz==0:
        return {"rule":"unsupported_vocabulary","intent":None,"confidence":0.0,
                "response":f"I don't have enough information to answer. Please ask a general {DOMAIN} health question."}
    probs=model.predict_proba([processed])[0]
    idx=int(np.argmax(probs));intent=model.classes_[idx];confidence=float(probs[idx])
    # Rule 4: low confidence
    if confidence<CONFIDENCE_THRESHOLD:
        return {"rule":"low_confidence","intent":intent,"confidence":confidence,
                "response":f"I'm not sure what you mean. Could you rephrase your {DOMAIN} question more specifically?"}
    # Rule 5: low relevance / unsupported question
    sim=float(vec.multiply(model.named_steps["tfidf"].transform(X_train)).sum(axis=1).max()) if False else None
    # use maximum cosine similarity: TF-IDF normalizes vectors
    train_vec=model.named_steps["tfidf"].transform(X_train)
    max_sim=float((vec @ train_vec.T).max())
    if max_sim<SIMILARITY_THRESHOLD:
        return {"rule":"out_of_domain","intent":None,"confidence":confidence,
                "response":f"This question appears outside my {DOMAIN} knowledge base. Please ask about symptoms, screening, risks, nutrition, activity, medication, prevention or basics."}
    # Rule 6: knowledge retrieval
    return {"rule":"knowledge_retrieval","intent":intent,"confidence":confidence,
            "response":KNOWLEDGE[DOMAIN][intent]+"\n\nEducational information only; not a diagnosis or personal treatment plan."}

print("Reasoning rules: emergency, personal medical decisions, unsupported, unknown vocabulary, low confidence, out of domain, knowledge retrieval.")

## 9. Integrated cognitive assistant — example inference

In [ ]:
for q in ["What foods support good health?", "How is diabetes diagnosed?" if DOMAIN=="diabetes" else "What is a cholesterol test?"]:
    result=assistant(q)
    print("\nUSER:",q,"\nRULE:",result["rule"],"| INTENT:",result["intent"],"| CONFIDENCE:",round(result["confidence"],3))
    print("ASSISTANT:",result["response"])

## 10. Required test cases — all intents, paraphrase, unsupported, low-confidence and emergency

In [ ]:
test_cases=[(f"intent_{intent}",qs[-1]) for intent,qs in CORPORA[DOMAIN].items()]
test_cases += [
("paraphrase","Could you explain ways to stay active and protect my health?"),
("unsupported","How can I treat a toothache?"),
("low_confidence","I have a vague question about something unrelated and unclear"),
("emergency","I have crushing chest pain and cannot breathe"),
("personal_medical_decision","What dose of medicine should I take?")
]
results=[]
for label,query in test_cases:
    output=assistant(query)
    results.append({"test":label,"query":query,"predicted_intent":output["intent"],"rule":output["rule"],"confidence":round(output["confidence"],3),"response":output["response"]})
display(pd.DataFrame(results))

## 11. Interactive Colab interface — ask multiple questions

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output
query_box=widgets.Textarea(placeholder="Enter one question per line...",description="Questions:",layout=widgets.Layout(width="95%",height="115px"))
ask_button=widgets.Button(description="Ask assistant",button_style="primary")
output_area=widgets.Output()
def on_ask(_):
    with output_area:
        clear_output()
        questions=[q.strip() for q in query_box.value.splitlines() if q.strip()]
        if not questions:
            print("Enter at least one question.")
            return
        for i,q in enumerate(questions,1):
            r=assistant(q)
            print(f"\n[{i}] You: {q}")
            print(f"Rule: {r['rule']} | Intent: {r['intent']} | Confidence: {r['confidence']:.2f}")
            print("Assistant:",r["response"])
ask_button.on_click(on_ask)
display(widgets.VBox([widgets.HTML(f"<h3>{DOMAIN.title()} Healthcare Cognitive Assistant</h3>"),query_box,ask_button,output_area]))

## Conclusion
This hybrid cognitive assistant combines a labelled corpus, taxonomy, ontology, NLP preprocessing, TF-IDF features, a Logistic Regression classifier, an intent-based knowledge base, and safety-first rules. Evaluation uses a held-out stratified test set. The interactive interface accepts multiple questions.

**Limitations:** The corpus is small and handcrafted; intent metrics do not measure clinical accuracy. Emergency keyword matching is incomplete. The assistant should not be used for diagnosis or treatment decisions.

